<a href="https://colab.research.google.com/github/MZiaAfzal71/Melting-Point-Prediction-of-Boronic-Acids/blob/main/Data%20Files/Scripts%20and%20Models/Extract_data_from_organoborons.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧾 Overview

This notebook automates the data extraction and preprocessing pipeline for boronic acid compounds from the Organoborons database. It systematically scrapes compound pages, retrieves each molecule’s name, InChIKey, melting point, and fetches corresponding SMILES strings from PubChem using the API. The data is then cleaned, standardized, and saved into a ready-to-use file — Cleaned_Boronic_Acids.xlsx — which serves as the input for descriptor generation and machine learning model development in subsequent notebooks.

## 🧩 Clone Repository and Navigate to Working Directory

The following cell clones the GitHub repository “Melting-Point-Prediction-of-Boronic-Acids” and changes the current working directory to the folder containing the data files, scripts, and pre-trained models used in this project.

In [1]:
!git clone https://github.com/MZiaAfzal71/Melting-Point-Prediction-of-Boronic-Acids
%cd Melting-Point-Prediction-of-Boronic-Acids/Data\ Files/Scripts\ and\ Models

Cloning into 'Melting-Point-Prediction-of-Boronic-Acids'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 85 (delta 12), reused 4 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 24.07 MiB | 11.80 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/Melting-Point-Prediction-of-Boronic-Acids/Data Files/Scripts and Models


#### This cell imports the essential Python libraries for data handling (pandas) and web scraping (requests, BeautifulSoup).

In [9]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

### This cell defines three helper functions:

* extract_boronic_pages() – Parses HTML content to extract all hyperlinks containing the word “data.”

* get_pubchem_info() – Retrieves the Canonical SMILES for a compound from PubChem using its InChIKey.

* process_melting_point() – Cleans melting point data by averaging ranges (e.g., “120–125”) or converting single values to floats.

In [11]:
def extract_boronic_pages(html_content):
    """
    Extracts all hyperlinks containing the keyword 'data' from the given HTML content.

    Parameters:
    - html_content (str): The HTML content of a webpage.

    Returns:
    - list: A list of extracted URLs that contain the word 'data'.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    hrefs = [a['href'] for a in soup.find_all('a', href=True)]  # Extract all href links
    filtered_hrefs = list(filter(lambda x: 'data' in x, hrefs))  # Keep only links containing 'data'

    return filtered_hrefs

def get_pubchem_info(inchikey):
    """
    Fetches the Canonical SMILES representation from PubChem using an InChIKey.

    Parameters:
    - inchikey (str): The InChIKey of the compound.

    Returns:
    - str: The Canonical SMILES string if found, otherwise an empty string.
    """
    url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{inchikey}/property/IUPACName,CanonicalSMILES/JSON'

    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad responses (4xx, 5xx)
        jstruct_res = response.json()
        return jstruct_res['PropertyTable']['Properties'][0].get('CanonicalSMILES', ' ')
    except (requests.exceptions.RequestException, KeyError, IndexError):
        return ' '  # Return empty string if any error occurs

# Function to convert melting point values
def process_melting_point(mp):
    if isinstance(mp, str) and '-' in mp:
        values = list(map(float, mp.split('-')))  # Convert range to list of floats
        return sum(values) / len(values)  # Compute average
    try:
        return float(mp)  # Convert single values directly to float
    except ValueError:
        return None  # Handle non-numeric cases

## This cell scrapes boronic acid data links from pages 1 to 7 of the organoborons.com website.
It collects all URLs containing “data,” stores them in a dictionary, converts them into a DataFrame, and then saves the results to an Excel file named Boronic_pages.xlsx inside the Excel Files folder.

In [10]:
# Dictionary to store extracted page links
bor_pages = {'pages': []}

# Loop through pages 1 to 7 and extract data links
for i in range(1, 8):
    url = f'https://organoborons.com/boronic-acids/page-{i}.html'
    response = requests.get(url)

    if response.status_code == 200:  # Check if the request was successful
        bor_pages['pages'] += extract_boronic_pages(response.text)
    else:
        print(f"Failed to fetch {url} (Status Code: {response.status_code})")

# Convert extracted links into a DataFrame and save to an Excel file
data = pd.DataFrame(bor_pages)
data.to_excel('Excel Files/Boronic_pages.xlsx', index=False)

print("Extraction complete. Data saved to 'Excel Files/Boronic_pages.xlsx'")

Extraction complete. Data saved to 'Excel Files/Boronic_pages.xlsx'


## This cell automates the data extraction and enrichment process for boronic acids:

It first reads the 1295 boronic acid page links from the previously saved Excel file and scrapes each webpage to extract the compound name, InChIKey, and melting point. Then, using the PubChem API, it fetches the corresponding SMILES representations for each compound.

The script saves intermediate results (without SMILES) and final results (with SMILES) into two Excel files:

* Boronic_Acids_IM.xlsx — Name, InChIKey, and Melting Point

* Boronic_Acids_SMILES.xlsx — same plus SMILES

⚠️ Since it processes all 1295 pages and queries PubChem for each compound, it may take around 25 minutes to complete.

In [12]:
# Define file paths
input_file = 'Excel Files/Boronic_pages.xlsx'
output_file1 = 'Excel Files/Boronic_Acids_IM.xlsx'
output_file2 = 'Excel Files/Boronic_Acids_SMILES.xlsx'

# Load Boronic pages from Excel
bor_file = pd.read_excel(input_file)

# Dictionary to store extracted Name, InChIKey, and Melting Point data
inch_melting = {'Name': [], 'InChIKey': [], 'Melting Point': []}

# Iterate over each page URL from the input file
for counter, pg in enumerate(bor_file['pages'], start=1):
    searchaddress = f'https://organoborons.com/{pg}'
    response = requests.get(searchaddress)

    if response.status_code == 200:
        soup = BeautifulSoup(response.text, 'html.parser')
        tables = soup.find_all('table')

        try:
            # Extract compound name from the first table
            inch_melting['Name'].append(tables[0].find('h1').text)

            # Extract data from the 4th table (index 3)
            rows = tables[3].find_all('tr')
            for row in rows:
                if 'Melting' in row.text:
                    melting_list = row.text.split()
                    inch_melting['Melting Point'].append(melting_list[2])  # Extract melting point value
                if 'InChIKey' in row.text:
                    inch_list = row.text.split()
                    inch_melting['InChIKey'].append(inch_list[1])  # Extract InChIKey

            print(f'{counter} entries processed successfully!')
        except (IndexError, AttributeError):
            print(f"Warning: Skipping {searchaddress} due to missing data.")
    else:
        print(f"Failed to retrieve {searchaddress} (Status Code: {response.status_code})")

# Ensure all extracted lists have the same length to prevent DataFrame misalignment
min_length = min(len(inch_melting['Name']), len(inch_melting['InChIKey']), len(inch_melting['Melting Point']))
inch_melting = {key: values[:min_length] for key, values in inch_melting.items()}

# Create DataFrame with an empty SMILES column
data = pd.DataFrame({
    'Name': inch_melting['Name'],
    'InChIKey': inch_melting['InChIKey'],
    'SMILES': [''] * len(inch_melting['Name']),  # Empty SMILES initially
    'Melting Point': inch_melting['Melting Point']
})

# Save the initial data (without SMILES) to an Excel file
data.to_excel(output_file1, index=False)

# Fetch SMILES for each compound using the PubChem API
for idx, inchi in enumerate(data['InChIKey']):
    data.loc[idx, 'SMILES'] = get_pubchem_info(inchi)
    print(f'{idx + 1}/{len(data)} compounds processed!')

# Save the final data with SMILES included
data.to_excel(output_file2, index=False)

print(f"Processing complete! Data saved to '{output_file2}'")

1 entries processed successfully!
2 entries processed successfully!
3 entries processed successfully!
4 entries processed successfully!
5 entries processed successfully!
6 entries processed successfully!
7 entries processed successfully!
8 entries processed successfully!
9 entries processed successfully!
10 entries processed successfully!
11 entries processed successfully!
12 entries processed successfully!
13 entries processed successfully!
14 entries processed successfully!
15 entries processed successfully!
16 entries processed successfully!
17 entries processed successfully!
18 entries processed successfully!
19 entries processed successfully!
20 entries processed successfully!
21 entries processed successfully!
22 entries processed successfully!
23 entries processed successfully!
24 entries processed successfully!
25 entries processed successfully!
26 entries processed successfully!
27 entries processed successfully!
28 entries processed successfully!
29 entries processed successf

## ✅ Explanation:
This cell loads the extracted boronic acid dataset, cleans it, and prepares it for descriptor (fingerprint) generation.

Specifically, it:

* Reads the file Boronic_Acids_SMILES.xlsx.

* Keeps only the relevant columns — Name, SMILES, and Melting Point.

* Converts all melting point values into numeric form using the process_melting_point() function.

* Removes any incomplete (missing) entries.

* Saves the cleaned dataset as Cleaned_Boronic_Acids.xlsx, which will later be used to compute molecular fingerprints and other descriptors.

In [13]:
# Load the Excel file
file_path = "Excel Files/Boronic_Acids_SMILES.xlsx"
df = pd.read_excel(file_path)

# Process the dataset
df_cleaned = df[['Name', 'SMILES', 'Melting Point']].copy()
df_cleaned['Melting Point'] = df_cleaned['Melting Point'].apply(process_melting_point)

# Remove rows with empty (NaN) values
df_cleaned.dropna(inplace=True)

# Save to a new Excel file
output_file = "Excel Files/Cleaned_Boronic_Acids.xlsx"
df_cleaned.to_excel(output_file, index=False)

print(f"Cleaned data saved to {output_file}")